In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

# -------------------------------------------------------------------------
# 1. Configuração de Hardware (GPU se disponível, senão CPU)
# -------------------------------------------------------------------------
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executando no dispositivo: {dispositivo}")

# -------------------------------------------------------------------------
# 2. Pipeline de Tensores e Transformações (A Ponte vista nas Aulas Anteriores)
# -------------------------------------------------------------------------
# transforms.ToTensor() converte a imagem (H, W, C) em [0, 255]
# para um tensor PyTorch (C, H, W) normalizado em [0.0, 1.0]
# Normalize() aplica a padronização (Z-score) por canal: (x - média) / desvio
transformacao = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# -------------------------------------------------------------------------
# 3. Aquisição Automática do Dataset CIFAR-10
# -------------------------------------------------------------------------
print("Baixando/Carregando a base de imagens CIFAR-10...")
treino_set = torchvision.datasets.CIFAR10(
    root="./dados_cifar", train=True, download=True, transform=transformacao
)
teste_set = torchvision.datasets.CIFAR10(
    root="./dados_cifar", train=False, download=True, transform=transformacao
)

# DataLoader organiza os tensores em lotes (Batch Size = 64)
treino_loader = torch.utils.data.DataLoader(treino_set, batch_size=64, shuffle=True)
teste_loader = torch.utils.data.DataLoader(teste_set, batch_size=64, shuffle=False)

classes = ('avião', 'carro', 'pássaro', 'gato', 'veado',
           'cão', 'sapo', 'cavalo', 'navio', 'caminhão')

# -------------------------------------------------------------------------
# 4. Definição da Arquitetura da CNN
# -------------------------------------------------------------------------
class CNNBasica(nn.Module):
    def __init__(self):
        super(CNNBasica, self).__init__()
        # Bloco Convolucional 1: Entrada 3 canais (RGB) -> Saída 16 filtros 3x3
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)  # 32x32 -> 16x16
        
        # Bloco Convolucional 2: Entrada 16 filtros -> Saída 32 filtros 3x3
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)  # 16x16 -> 8x8
        
        # Camadas Densas (Classificador Final)
        # 32 canais * 8 altura * 8 largura = 2048 entradas
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)  # 10 saídas correspondentes às 10 classes

    def forward(self, x):
        # Passagem direta (Forward pass)
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        
        # Achatamento (Flatten) de (N, 32, 8, 8) para (N, 2048)
        x = x.view(x.size(0), -1)
        
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

modelo = CNNBasica().to(dispositivo)

# -------------------------------------------------------------------------
# 5. Critério de Perda e Otimizador
# -------------------------------------------------------------------------
criterio = nn.CrossEntropyLoss()
otimizador = optim.Adam(modelo.parameters(), lr=0.001)

# -------------------------------------------------------------------------
# 6. Loop de Treinamento Didático
# -------------------------------------------------------------------------
epocas = 3
print("\n--- INICIANDO TREINAMENTO ---")

for epoca in range(epocas):
    modelo.train()
    perda_acumulada = 0.0
    
    for i, (imagens, rotulos) in enumerate(treino_loader):
        imagens, rotulos = imagens.to(dispositivo), rotulos.to(dispositivo)
        
        # Zera os gradientes acumulados
        otimizador.zero_grad()
        
        # Inferência (Forward)
        predicoes = modelo(imagens)
        
        # Cálculo do erro
        perda = criterio(predicoes, rotulos)
        
        # Retropropagação (Backward) e atualização dos pesos
        perda.backward()
        otimizador.step()
        
        perda_acumulada += perda.item()
        if (i + 1) % 200 == 0:
            print(f"[Época {epoca + 1}, Lote {i + 1}] Perda média: {perda_acumulada / 200:.4f}")
            perda_acumulada = 0.0

# -------------------------------------------------------------------------
# 7. Avaliação da Acurácia no Conjunto de Teste
# -------------------------------------------------------------------------
modelo.eval()
acertos = 0
total = 0

with torch.no_grad():
    for imagens, rotulos in teste_loader:
        imagens, rotulos = imagens.to(dispositivo), rotulos.to(dispositivo)
        saidas = modelo(imagens)
        _, preditos = torch.max(saidas.data, 1)
        total += rotulos.size(0)
        acertos += (preditos == rotulos).sum().item()

acuracia = 100.0 * acertos / total
print(f"\nAcurácia do modelo no conjunto de teste: {acuracia:.2f}%")

Executando no dispositivo: cpu
Baixando/Carregando a base de imagens CIFAR-10...


100.0%



--- INICIANDO TREINAMENTO ---
[Época 1, Lote 200] Perda média: 1.7718
[Época 1, Lote 400] Perda média: 1.4640
[Época 1, Lote 600] Perda média: 1.3734
[Época 2, Lote 200] Perda média: 1.1983
[Época 2, Lote 400] Perda média: 1.1445
[Época 2, Lote 600] Perda média: 1.1103
[Época 3, Lote 200] Perda média: 1.0245
[Época 3, Lote 400] Perda média: 0.9953
[Época 3, Lote 600] Perda média: 0.9703

Acurácia do modelo no conjunto de teste: 65.08%
